# Analyse trail — Afficher la trace de la course sur une carte

Dans ce notebook qui fait suite au 2 précédents 
- lire les données
- analyser les données de terrain,

on s'intéresse à l'affichage de la trace GPS contenue dans le fichier FIT (latitude, longiture) sur une carte à laquelle on ajoute quelques informations comme les ravitos, les points de départ et d'arrivée...


**Pré-requis :**

Cette fois, on va avoir besoin d'une librairie supplémentaire qui s'appelle `Folium`, très utilisée dans le monde de la télédétection, notamment. 


```bash
pip install fitparse pandas numpy matplotlib folium

```

**Remarque**

Dans les premiers notebooks, j'avais volontairement laissé le code complet dans les cellules. Cette fois, c'est fini, l'essentiel des codes est déplacé dans le fichier annexe `trail_analysis_pub.py`. Les notebooks qui contiennent de plus en plus de cellule d'analyses des données sont allégés. 

N'oublie pas de récupérer le fichier `trail_analysis_pub.py` en local pour faire tourner le notebook.

### 
### © Gregory Sainton
### 
- Notebook        : Analyse d'une séance de trail/course 
- Auteur          : Grégory Sainton <trinity4trail@proton.me>
- Dépôt           : https://github.com/gregs1t/trail
- Date de création: 2025-12
- Dernière modif. : 2026-04
- Version         : 1.0
---
- Licence         : CC BY-NC-SA 4.0
-                   https://creativecommons.org/licenses/by-nc-sa/4.0/

>    Vous êtes libre de partager et d'adapter ce travail, à condition de :
>     · citer l'auteur (BY)
>     · ne pas en faire un usage commercial (NC)
>     · redistribuer sous la même licence (SA)

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # trail_analysis gère ses propres filtres

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trail_analysis_pub import (
    load_fit, clean_df, plot_map,
    compute_slope, compute_dplus_dminus, compute_gap,
    segment_updown, classify_walk_run, plot_dashboard,
    plot_walk_by_slope_sections)

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
print("Imports OK.")

## Paramètres utilisateur

**Seule cellule à modifier.**

In [ ]:
TARGET = { 
         'label':      'macourse 2026',
         'fit_path':   'mon_fichier_a_moi.fit',
         'ravito_km':  [25.4, 37.7, 47.7, 53.8, 78.3],
         'ravito_nom': ['Buc', 'Chaville', 'Saint Philippe', 'Marcel Bec', 'Saint Cloud'],
    }

# --- Fichier FIT ---
FIT_PATH = TARGET["fit_path"]

# --- Physiologie ---
FC_MAX    = 453     # FC max réelle (bpm)
FC_MIN    = 22      # FC repos (bpm)
POIDS_KG  = 138.0    # masse corporelle (kg)

# --- Ravitaillements ---
RAVITO_KM  = TARGET["ravito_km"]
RAVITO_NOM = TARGET["ravito_nom"]

# --- Calcul de la pente ---
WINDOW_M     = 100.0    # fenêtre glissante en mètres
UP_THR       = +3.0     # seuil montée (%)
DOWN_THR     = -3.0     # seuil descente (%)
MIN_SEG_M    = 200.0    # longueur minimale de segment (m)

# --- Seuils marche / course ---
WALK_THR_KMH = 5.0      # seuil vitesse (km/h)
WALK_THR_CAD = 140.0    # seuil cadence (pas/min)

---
## 1. Chargement et nettoyage

In [ ]:
df_raw = load_fit(FIT_PATH)
df, ALT_COL = clean_df(df_raw)
df["slope_pct"] = compute_slope(df, WINDOW_M)
df = compute_gap(df)
df = segment_updown(df, UP_THR, DOWN_THR, MIN_SEG_M)

if "cadence" in df.columns:
    df = classify_walk_run(df, WALK_THR_KMH, WALK_THR_CAD)

dplus, dminus = compute_dplus_dminus(df)

print(f"Points       : {len(df)}")
print(f"Distance     : {df['dist_m'].max()/1000:.1f} km")
print(f"Durée        : {df['time_h'].max():.2f} h")
print(f"D+           : {dplus:.0f} m")
print(f"D-           : {dminus:.0f} m")
if "heart_rate" in df.columns:
    print(f"FC moy / max : {df['heart_rate'].mean():.0f} / {df['heart_rate'].max():.0f} bpm")

---
## 2. Cartographie de la trace

In [ ]:
# Carte colorée par FC (changer color_col selon besoin : "speed_kmh", "slope_pct", None)
# animated=True      : trace animée AntPath (sens de course)
# show_elevation=True : profil altimétrique en incrustation (coin inférieur gauche)
m = plot_map(
    df,
    ravito_km=RAVITO_KM,
    ravito_nom=RAVITO_NOM,
    color_col="heart_rate",
    cmap_name="RdYlGn_r",
    animated=True,
    show_elevation=True,
)
m   # affiche la carte interactive dans Jupyter

---
## 3. Profil altimétrique

In [ ]:
x = df["dist_m"] / 1000.0
y = df["alt_m"]

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(x, y, color="brown", linewidth=2)
ax.fill_between(x, y, y.min(), color="brown", alpha=0.3)

for km, nom in zip(RAVITO_KM, RAVITO_NOM):
    idx = (df["dist_m"] / 1000.0 - km).abs().idxmin()
    ax.axvline(km, color="grey", linestyle=":", alpha=0.6)
    ax.text(km, ax.get_ylim()[1] * 0.95, nom,
            fontsize=7, rotation=90, va="top", ha="right", color="grey")

ax.set_xlabel("Distance (km)")
ax.set_ylabel("Altitude (m)")
ax.set_title("Profil altimétrique + ravitaillements")
fig.tight_layout()
plt.show()

---
## 4. Dashboard FC / allure / température

In [ ]:
plot_dashboard(df, FC_MIN, FC_MAX)

---
## Marche par classe de pente

In [ ]:
SLOPE_BINS   = [-30, -10, -3, 3, 10, 20, 40]
SLOPE_LABELS = ["< -10%", "-10/-3%", "plat", "+3/+10%", "+10/+20%", "> +20%"]

if "cadence" in df.columns:
    plot_walk_by_slope_sections(
        df, slope_bins=SLOPE_BINS, slope_labels=SLOPE_LABELS,
        ravito_km=RAVITO_KM,
    )
else:
    print("Pas de cadence.")

# Et si on veut une version avec des segments identiques :
# Par exemple, 2 segments pour distinguer les deux moitiés de la course.
if "cadence" in df.columns:
    plot_walk_by_slope_sections(
        df, slope_bins=SLOPE_BINS, slope_labels=SLOPE_LABELS,
        n_segments=2)
else:
    print("Pas de cadence.")